# BRIEF PRO TVORBU index.html
## Prague Stay Management — Big Data Project

---

## CO JE index.html

Průvodní HTML dokument pro odevzdání projektu profesorovi.
Jednoduchá HTML stránka (inline CSS, žádné externí soubory).
Struktura dle zadání profesora — 6 sekcí.
Styl: stručně, věcně, odrážky, tabulky, grafy/obrázky kde relevantní.

---

## SEKCE 1 — Představení firmy a cílů (KPI)

### Firma
- Název: **Prague Stay Management**
- Doména: správa a krátkodobý pronájem bytů v Praze (Property Management)
- Zákazníci/uživatelé: management firmy

### Business questions
1. Umožňuje blízkost stanice metra (≤500 m) účtovat prémiovou cenu a generovat vyšší příjem?
2. Ve kterých měsících přicházíme o nejvíce peněz kvůli nízké obsazenosti a stornům?
3. Vyplatí se mít velké byty (6+ osob) v portfoliu — jsou kapacitně a finančně efektivní?

### KPI

| KPI | Definice | Vzorec | Jednotka | Baseline | Cíl | Frekvence |
|-----|----------|--------|----------|----------|-----|-----------|
| #1 RevPAR × metro | Roční příjem na byt podle vzdálenosti od metra | estimated_revenue_l365d | CZK/rok | Medián 165 000 CZK | Identifikovat prémii u metra | Roční |
| #2 Vacancy Loss Rate | Podíl prázdných dní v kalendáři | (1 - AVG(occupied)) × 100 | % | 49.3 % | Identifikovat rizikové měsíce | Měsíční |
| #3 Capacity Utilization | Využití kapacity bytu | AVG(guest_count / accommodates) × 100 | % | 55 % (velké), 75.9 % (malé) | Optimalizovat portfolio | Na rezervaci |

### Stakeholdeři a rozhodnutí
- Management firmy rozhoduje o: rozšíření portfolia u metra, zavedení nevratných tarifů, zastoupení velkých bytů
- Rozhodnutí umožněná KPI: kdy zpřísnit storno podmínky, zda investovat do bytů u metra, jak optimalizovat velikost bytů

### Předpoklady a omezení
- Data z Inside Airbnb — scraping, ne transakční data
- `calendar.price` 100 % NaN — ceny pouze z listings
- Outliers v ceně ořezány na 99. percentil (~21 823 CZK) — možné mírné podhodnocení prémiového segmentu
- Data calendar pokrývají září 2025 – září 2026 — říjen a září mají disproporční počty záznamů

---

## SEKCE 2 — Technické řešení

### Zdrojová data
- Inside Airbnb (reálná): listings, calendar, metro_stations
- Syntetická: bookings.csv, expenses.csv

### Integrace
- Batch zpracování, jednorázový import
- Soubory: .gz (komprimované CSV), .csv
- Základní transformace: čištění, imputace, Haversine výpočet, run-length encoding

### Uložení
- Lokální: DuckDB (`database/prague_stay.duckdb`)
- Processed data: `data/processed/`
- Synthetic data: `data/synthetic/`

### Zpracování — pipeline
1. Čištění → 2. Obohacení (metro vzdálenost) → 3. Generování syntetiky → 4. Načtení do DuckDB → 5. KPI analýza

### Škálování
- Při ×10 dat: DuckDB zvládne, calendar by byl ~40M řádků — zvážit particionování podle měsíce
- Near-real-time: nahradit batch import streamingem (Kafka), DuckDB → ClickHouse

### Diagramy — popsat nebo nakreslit v HTML

**High-level architektura:**
```
[Inside Airbnb] → listings.csv, calendar.csv
[AI generátor]  → metro_stations.csv
        ↓
[Python / Pandas] — čištění, transformace
        ↓
[DuckDB] — 5 tabulek
        ↓
[Jupyter Notebooks] — KPI analýza
        ↓
[index.html] — prezentace výsledků
```

**Datový tok:**
```
listings_csv.gz → listings_enriched.csv → DuckDB: listings
calendar_csv.gz → calendar_clean.csv   → DuckDB: calendar
metro_stations.csv                     → DuckDB: metro_stations
calendar (run-length encoding)         → bookings.csv → DuckDB: bookings
bookings (confirmed only)              → expenses.csv → DuckDB: expenses
```

---

## SEKCE 3 — Data Strategy & Governance

### Lifecycle
- Retenční politika: raw data archivovat, processed přegenerovat při novém scrapu
- Fáze: Ingestion → Cleaning → Enrichment → Synthetic generation → Analysis → Presentation

### Role a odpovědnosti (RACI)

| Aktivita | Data Engineer | Data Analyst | Management |
|----------|--------------|--------------|------------|
| Sběr a čištění dat | R | C | I |
| Generování syntetiky | R | C | I |
| KPI analýza | C | R | I |
| Rozhodnutí o portfoliu | I | C | R |
| Kvalita dat | R | C | I |
| Prezentace výsledků | C | R | A |

### Kvalita dat

| Dimenze | Metrika | Práh |
|---------|---------|------|
| Úplnost | % NaN v price_czk | < 15 % před imputací ✅ (13.2 %) |
| Konzistence | FK porušení v DuckDB | 0 ✅ |
| Přesnost | Logické anomálie (ceny < 0, accommodates = 0) | 0 ✅ |
| Aktuálnost | Rozsah dat calendar | září 2025 – září 2026 ✅ |

### Etika a rizika
- **Bias**: data z Inside Airbnb neobsahují odmítnuté rezervace — selection bias
- **Syntetická data**: bookings.csv a expenses.csv jsou generované podle reálných vzorců z EDA (sezónnost, distribuce hostů, náklady na úklid). KPI #2 cancel rate a KPI #3 capacity utilization vycházejí z těchto dat — reprezentují modelovaný scénář. Všechny ostatní metriky vycházejí z reálných dat Inside Airbnb.
- **Reprezentativnost**: Praha 1 tvoří 35 % dat — výsledky mohou být zkresleny centrem
- **Outliers**: ořezání na 99. percentil mírně podhodnocuje prémiový segment
- **Omezení interpretace**: korelace metro × příjem byla ověřena robustness checkem uvnitř čtvrtí — efekt je reálný, ne jen proxy pro centrum

---

## SEKCE 4 — Data a prezentace

### Přehled datasetů

| Dataset | Zdroj | Typ | Rozsah | Granularita | Klíče | Licence |
|---------|-------|-----|--------|-------------|-------|---------|
| listings_csv.gz | Inside Airbnb | Reálná | 10 834 řádků, 79 sloupců | 1 řádek = 1 byt | id | CC BY 4.0 |
| calendar_csv.gz | Inside Airbnb | Reálná | 3 954 411 řádků, 7 sloupců | 1 řádek = 1 byt × 1 den | listing_id + date | CC BY 4.0 |
| metro_stations.csv | AI generováno | Syntetická | 61 řádků, 5 sloupců | 1 řádek = 1 stanice | station_id | — |
| bookings.csv | Odvozeno z calendar + syntetický status | Semi-syntetická | 74 671 řádků | 1 řádek = 1 rezervace | booking_id | — |
| expenses.csv | Syntetická | Syntetická | 63 765 řádků | 1 řádek = 1 rezervace (confirmed) | expense_id | — |

### Syntetická data — metodika

**metro_stations.csv:**
- Generováno AI (Claude) na základě reálné sítě metra Praha
- Linky A (17 stanic), B (24 stanic), C (20 stanic)
- Validováno porovnáním s veřejnými zdroji

**bookings.csv:**
- Základ: consecutive dny `available = f` z calendar → run-length encoding → 1 bронирование
- `guest_count`: useknuté normální rozdělení, moda závisí na kapacitě (≤2: moda=accommodates, ≤5: moda=accommodates×0.75, 6+: moda=accommodates×0.55)
- `price_per_night`: ±15 % šum kolem reálné ceny z listings
- `status`: sezónně vážený cancel rate (leden/únor/listopad: 20 %, léto: 10 %, ostatní: 13–18 %)
- `MAX_STAY = 30 dní` — cap pro krátkodobý pronájem
- Validace: guest_count ≤ accommodates pro všechny záznamy ✅

**expenses.csv:**
- Pouze pro confirmed bookings
- `cleaning_cost`: nelineární škála podle bedrooms (0: 600, 1: 900, 2: 1400, 3: 2000, 4+: 2600+ CZK) + šum ±15 %
- `maintenance_cost`: náhodná událost s pravděpodobností 5 %
- Omezení: "příliš hezká" data eliminována šumem ±15 %

### Předzpracování
- Odstraněno 57 sloupců (z 79), včetně 3 sloupců s 100 % NaN
- `price`: string → float, odstranění `$` a `,` (artefakt exportu Inside Airbnb, hodnoty v CZK)
- Outliers: horní práh = 99. percentil (21 823 CZK), dolní práh = Entire home/apt < 400 CZK → NaN
- Imputace ceny: 3-stupňová (neighbourhood + room_type + accommodates → neighbourhood + room_type → globální medián)
- Haversine vzdálenost: vektorizovaný výpočet N×M matice (10 834 × 61 stanic)
- `near_metro`: binární příznak pro vzdálenost ≤ 500 m

### DuckDB schéma

```
metro_stations (station_id PK)
    ↑
listings (id PK, nearest_metro_id FK)
    ↑                    ↑
calendar              bookings (booking_id PK, listing_id FK)
(listing_id FK,           ↑
 date)                expenses (expense_id PK, booking_id FK, listing_id FK)
```

### Klíčové výsledky EDA (pro grafy v HTML)

**Listings:**
- 10 834 bytů, 82 % Entire home/apt
- Medián ceny: 2 086 CZK/noc
- Praha 1: 35 % všech nabídek
- Velké byty (6+): 20 % portfolia
- near_metro = True: 65.9 % bytů

**Calendar:**
- Průměrná obsazenost: 50.7 %
- Peak: září (64.6 %), minimum: listopad (37.3 %) a únor (37.4 %)

**Korelace:**
- estimated_occupancy × estimated_revenue: r = 0.54
- price_czk × accommodates: r = 0.09 (velké byty nejsou nutně dražší)

---

## SEKCE 5 — Diskuze, závěry a doporučení

### KPI #1 — RevPAR × metro
- Byty u metra (≤500 m): medián 211 395 CZK/rok
- Byty mimo metro (>500 m): medián 118 932 CZK/rok
- **Rozdíl: 78 %**
- Robustness check uvnitř čtvrtí potvrdil že efekt je reálný (Praha 3: rozdíl 40 %)
- Doporučení: rozšiřovat portfolio prioritně o byty ≤500 m od metra, zejména Praha 2 a Praha 3

### KPI #2 — Vacancy Loss Rate
- Nejrizikovější měsíce: leden (62 %), únor (63 %), listopad (63 %)
- Cancel rate v zimě: 20 %, v létě: 9–10 %
- Doporučení: zavést nevratné tarify pro říjen–únor, cenové pobídky v zimě

### KPI #3 — Capacity Utilization
- Malé byty (≤5): utilization 75.9 %, revenue per accommodates 78 204 CZK
- Velké byty (6+): utilization 55.0 %, revenue per accommodates 63 331 CZK
- **KLÍČOVÝ INSIGHT:** absolutní marže velkých bytů je o 55 % vyšší (26 247 vs 16 893 CZK)
- RevPASH: malé 9 678 CZK/host, velké 8 111 CZK/host (–16 %)
- Závěr: velké byty nejsou nevýhodné — problém je obsazenost, ne velikost
- Doporučení: nezavrhovat velké byty, ale cíleně marketovat na skupiny a rodiny

### Limity řešení
- Selection bias Inside Airbnb — neobsahuje odmítnuté rezervace
- Praha 1 dominuje datasetu (35 %) — možné zkreslení
- Ořezání outlierů mírně podhodnocuje prémiový segment
- Syntetická data (bookings, expenses) reprezentují modelovaný scénář — viz sekce 3

### Budoucnost a AI
- Dynamic pricing model (ML) na základě metro vzdálenosti, sezóny a kapacity
- Automatizace: nový scraping → přegenerování processed dat → aktualizace DuckDB
- Rizika: model může reprodukovat bias z dat, nutná lidská kontrola doporučení

---

## SEKCE 6 — Metodika a postup

### Workflow
1. Business Understanding — definice KPI a business questions
2. Data Understanding — EDA (notebook 01)
3. Data Preparation — čištění, syntetika, DuckDB (notebook 02)
4. Modeling — KPI analýza, agregace (notebook 03)
5. Evaluation — závěry, limity (notebook 03 + index.html)
6. Deployment — DuckDB + index.html

### Rozdělení rolí
- Doplnit Jmena a co kdo delal

### Nástroje
- Python 3.11, Pandas 2.2, NumPy, Matplotlib, Seaborn, Folium
- DuckDB 1.1
- Jupyter Notebook
- Generativní AI: Claude (Anthropic)

### AI použití
- Generování metro_stations.csv (validováno s veřejnými zdroji)
- Asistence při psaní kódu pro čištění, Haversine, run-length encoding
- Kontrola faktů: všechny výsledky ověřeny spuštěním kódu a porovnáním s reálnými daty
- Prompty zaměřeny na: strukturu projektu, metodiku imputace, DuckDB schéma, interpretaci výsledků

### Zdroje
- Inside Airbnb: http://insideairbnb.com
- DuckDB dokumentace: https://duckdb.org/docs
- Haversine formula: https://en.wikipedia.org/wiki/Haversine_formula
